# Color Palettes Demo
Explore and compare all built-in colormaps available in `PALETTES` across regular seismic gathers, F-K spectra, and 2D velocity models.


In [ ]:
%pip install bokeh segyio

# *** Install pyseiskit from PyPI
# %pip install pyseiskit
# *** Install pyseiskit for local development
# %pip install -e ..
!uv pip install -e .. --reinstall


## 1. Data Setup
Load a shot gather, compute its F-K spectrum, and load a 2D velocity model to test the palettes across all three data types.


In [ ]:
import segyio
import numpy as np
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import column, gridplot
from bokeh.models import LinearColorMapper, ColorBar
from seismicReader import readSeismicFile
from pyseiskit import fk, gain, normalizeTraces, PALETTES

output_notebook()

# 1. Load specific shot gather (fldr=50)
gatherFilename = 'tac-204RL239.su'
gatherData, traceOffsets, timeSamples = readSeismicFile(
    gatherFilename,
    gatherKey=segyio.TraceField.FieldRecord,
    gatherIndex=49
)

intervalTimeSamples = 0.004
if len(timeSamples) > 1:
    intervalTimeSamples = timeSamples[1] - timeSamples[0]
intervalSpaceSamples = 50.0  # dx=50m

# Apply TPOW and trace-by-trace max normalization to balance visual energy
gatherData = gain.applyTimePowerGain(gatherData, 0.7, intervalTimeSamples)
gatherData = normalizeTraces(gatherData, method='max')

# 2. Compute F-K amplitude spectrum
wavenumbers, frequencies, complexFKSpectrum = fk.computeFKSpectrum(
    gatherData,
    intervalTimeSamples,
    intervalSpaceSamples
)
amplitudeFKSpectrum = np.abs(complexFKSpectrum)

# 3. Load 2D velocity model
velocityFilename = 'modPGCto_vel2_5x5_topoElev_verror.su'
velocityModel, velocityTraceOffsets, depthSamples = readSeismicFile(velocityFilename)

print(f"Available palettes ({len(PALETTES)}): {list(PALETTES.keys())}")


## 2. Color Ramp Swatches
Preview the continuous 256-step gradient of every palette exposed in `PALETTES`.


In [ ]:
gradientRamp = np.linspace(0.0, 1.0, 256).reshape(1, 256)
swatchFigures = []

for paletteName, paletteColors in PALETTES.items():
    swatchFigure = figure(
        width=700,
        height=55,
        title=f"{paletteName} ({len(paletteColors)} colors)",
        x_range=(0, 1),
        y_range=(0, 1),
        toolbar_location=None
    )
    colorMapper = LinearColorMapper(palette=paletteColors, low=0.0, high=1.0)
    swatchFigure.image(image=[gradientRamp], x=0, y=0, dw=1, dh=1, color_mapper=colorMapper)
    swatchFigure.xaxis.visible = False
    swatchFigure.yaxis.visible = False
    swatchFigure.xgrid.grid_line_color = None
    swatchFigure.ygrid.grid_line_color = None
    swatchFigures.append(swatchFigure)

show(column(*swatchFigures))


## 3. Seismic Gather Across Palettes
Compare how each palette renders a normalized shot gather in the time-offset domain.


In [ ]:
def plotGatherWithPalette(gatherData, timeSamples, traceOffsets, paletteName, width=320, height=420):
    plotFigure = figure(
        width=width,
        height=height,
        title=paletteName,
        x_axis_label="Offset (m)",
        y_axis_label="Time (s)",
        x_range=(traceOffsets[0], traceOffsets[-1]),
        y_range=(timeSamples[-1], timeSamples[0])
    )

    colorMapper = LinearColorMapper(palette=PALETTES[paletteName], low=-1.0, high=1.0)
    plotFigure.image(
        image=[gatherData],
        x=traceOffsets[0],
        y=timeSamples[0],
        dw=traceOffsets[-1] - traceOffsets[0],
        dh=timeSamples[-1] - timeSamples[0],
        color_mapper=colorMapper
    )

    plotFigure.add_layout(ColorBar(color_mapper=colorMapper), 'right')
    plotFigure.xgrid.grid_line_color = None
    plotFigure.ygrid.grid_line_color = None
    return plotFigure

gatherPlots = [
    plotGatherWithPalette(gatherData, timeSamples, traceOffsets, paletteName)
    for paletteName in PALETTES
]

show(gridplot(gatherPlots, ncols=3))


## 4. F-K Spectrum Across Palettes
Compare how each palette highlights spectral energy and coherent noise in the F-K domain.


In [ ]:
def plotFkSpectrumWithPalette(amplitudes, wavenumbers, frequencies, paletteName, width=320, height=420):
    plotFigure = figure(
        width=width,
        height=height,
        title=paletteName,
        x_axis_label="Wavenumber (cycles/m)",
        y_axis_label="Frequency (Hz)",
        x_range=(wavenumbers[0], wavenumbers[-1]),
        y_range=(125, 0)
    )

    colorMapper = LinearColorMapper(palette=PALETTES[paletteName])
    plotFigure.image(
        image=[amplitudes],
        x=wavenumbers[0],
        y=frequencies[0],
        dw=wavenumbers[-1] - wavenumbers[0],
        dh=frequencies[-1] - frequencies[0],
        color_mapper=colorMapper
    )

    plotFigure.add_layout(ColorBar(color_mapper=colorMapper), 'right')
    plotFigure.xgrid.grid_line_color = None
    plotFigure.ygrid.grid_line_color = None
    return plotFigure

fkPlots = [
    plotFkSpectrumWithPalette(amplitudeFKSpectrum, wavenumbers, frequencies, paletteName)
    for paletteName in PALETTES
]

show(gridplot(fkPlots, ncols=3))


## 5. Velocity Model Across Palettes
Compare how each palette renders a 2D velocity model (`modPGCto_vel2_5x5_topoElev_verror.su`).


In [ ]:
def plotVelocityModelWithPalette(velocityData, depthSamples, traceOffsets, paletteName, width=450, height=300):
    plotFigure = figure(
        width=width,
        height=height,
        title=paletteName,
        x_axis_label="Trace Index",
        y_axis_label="Depth (km)",
        x_range=(traceOffsets[0], traceOffsets[-1]),
        y_range=(depthSamples[-1], depthSamples[0])
    )

    colorMapper = LinearColorMapper(palette=PALETTES[paletteName])
    plotFigure.image(
        image=[velocityData],
        x=traceOffsets[0],
        y=depthSamples[0],
        dw=traceOffsets[-1] - traceOffsets[0],
        dh=depthSamples[-1] - depthSamples[0],
        color_mapper=colorMapper
    )

    plotFigure.add_layout(ColorBar(color_mapper=colorMapper, title="m/s"), 'right')
    plotFigure.xgrid.grid_line_color = None
    plotFigure.ygrid.grid_line_color = None
    return plotFigure

velocityPlots = [
    plotVelocityModelWithPalette(velocityModel, depthSamples, velocityTraceOffsets, paletteName)
    for paletteName in PALETTES
]

show(gridplot(velocityPlots, ncols=2))
